# Chinese and Hindi De-identification Tour

**Audience:** Developers evaluating OpenMed with Simplified Chinese, Hindi, or code-mixed Hinglish clinical text.

**Prerequisites:** Run this notebook from an OpenMed source checkout with the project environment installed (`uv pip install -e ".[dev]"`). All notes and identifiers below are fabricated test data.

**Learning goals:** By the end, you can run the two companion scripts, inspect structured PII entities, save UTF-8 redacted notes, and enforce a zero-leak assertion for known synthetic identifiers.


## Outline

1. Prepare a deterministic, no-download tutorial loader.
2. De-identify and save a Simplified Chinese clinical note.
3. De-identify and save Hindi and Hinglish clinical notes.
4. Check the fail-closed leakage invariant and try an extension.


## 1. Deterministic setup

The scripts normally use shipped checkpoints. For a notebook that executes without downloading model artifacts, this tour supplies an empty token-classification loader. The same real `deidentify()` pipeline still runs: the Chinese example uses explicit tutorial terms, while the India policy enables the ABDM recognizer for Aadhaar and ABHA and the safety sweep covers the reserved email.


In [1]:
from __future__ import annotations

import importlib.util
import json
import logging
import sys
import warnings
from pathlib import Path
from tempfile import TemporaryDirectory
from typing import Any

warnings.filterwarnings("ignore", message="IProgress not found.*")
logging.getLogger("openmed.core.models").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)


def find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "examples/deid_chinese_clinical_note.py").is_file():
            return candidate
    raise RuntimeError("Run this notebook from an OpenMed source checkout.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def load_example(module_name: str, relative_path: str) -> Any:
    spec = importlib.util.spec_from_file_location(
        module_name, REPO_ROOT / relative_path
    )
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load {relative_path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module


prior_logging_disable = logging.root.manager.disable
logging.disable(logging.CRITICAL)
try:
    chinese = load_example(
        "openmed_example_chinese", "examples/deid_chinese_clinical_note.py"
    )
    hindi = load_example(
        "openmed_example_hindi", "examples/deid_hindi_hinglish_note.py"
    )
finally:
    logging.disable(prior_logging_disable)


class NoDownloadTokenClassificationPipeline:
    tokenizer = None

    def __call__(self, inputs: Any, **_: Any) -> list[Any]:
        return [[] for _ in inputs] if isinstance(inputs, list) else []


class NoDownloadLoader:
    config = None

    def create_pipeline(self, *_: Any, **__: Any) -> Any:
        return NoDownloadTokenClassificationPipeline()

    def get_max_sequence_length(self, *_: Any, **__: Any) -> None:
        return None


workspace = TemporaryDirectory()
output_root = Path(workspace.name)
offline_loader = NoDownloadLoader()
print("Tutorial mode: no model downloads; synthetic local rules enabled.")


Tutorial mode: no model downloads; synthetic local rules enabled.


## 2. Simplified Chinese clinical note

The Chinese script uses the accepted `zh` route for Chinese normalization and built-in patterns, the shipped generic compact detector, and explicit local terms for the tutorial's fabricated values. The `china_pipl` policy controls replacement and masking actions. The route does not claim a dedicated trained Chinese checkpoint.


In [2]:
chinese_output = output_root / "chinese_note_redacted.txt"
chinese_result = chinese.run_chinese_deidentification(
    chinese_output,
    loader=offline_loader,
)
chinese.assert_synthetic_identifiers_removed(chinese_result.deidentified_text)

print(chinese_result.deidentified_text)
print(json.dumps(chinese.structured_entities(chinese_result), ensure_ascii=False, indent=2))
print("Saved UTF-8 output:", chinese_output.is_file())


【完全虚构的示例】患者伍淑兰，病历号 623721195208297490，出生日期 [DATE]，身份证号 211380199602268733。联系电话 +45 22 1449 5285，电子邮箱 guiyingye@example.test，住址 新岚路739号，邮编778916。患者因咳嗽复诊，原金凤医生建议继续观察。
[
  {
    "label": "PERSON",
    "text": "李晓雯",
    "start": 11,
    "end": 14,
    "confidence": 0.95,
    "replacement": "伍淑兰",
    "sources": [
      "safety_sweep"
    ]
  },
  {
    "label": "ID_NUM",
    "text": "ZH-DEMO-708",
    "start": 19,
    "end": 30,
    "confidence": 1.0,
    "replacement": "623721195208297490",
    "sources": [
      "custom:deny"
    ]
  },
  {
    "label": "DATE",
    "text": "1990年2月3日",
    "start": 36,
    "end": 45,
    "confidence": 1.0,
    "replacement": "[DATE]",
    "sources": [
      "custom:deny"
    ]
  },
  {
    "label": "ID_NUM",
    "text": "110108198503150018",
    "start": 51,
    "end": 69,
    "confidence": 0.95,
    "replacement": "211380199602268733",
    "sources": [
      "safety_sweep"
    ]
  },
  {
    "label": "PHONE",
    "text": "+86 10 5555 0708",
    "start"

The placeholders show that all eight deliberately embedded identifiers were removed. The structured rows preserve offsets, labels, detector provenance, and replacement values for downstream review.


## 3. Hindi and code-mixed Hinglish notes

The Hindi companion script runs the same flow twice with the shipped compact Hindi checkpoint: once on Devanagari Hindi and once on Latin-script Hinglish. The `india_dpdp_act` policy enables the validated ABDM recognizer for the fabricated Aadhaar- and ABHA-format values, while local rules make the names, medical-record number, and spaced phone display form deterministic.


In [3]:
hindi_output_dir = output_root / "hindi_hinglish"
hindi_results = hindi.run_hindi_hinglish_deidentification(
    hindi_output_dir,
    loader=offline_loader,
)

for note_name, result in hindi_results.items():
    hindi.assert_synthetic_identifiers_removed(note_name, result.deidentified_text)
    rows = hindi.structured_entities(result)
    saved_path = hindi_output_dir / f"{note_name}_note_redacted.txt"
    print(f"--- {note_name.title()} ---")
    print(result.deidentified_text)
    print(json.dumps(rows, ensure_ascii=False, indent=2))
    print("Saved UTF-8 output:", saved_path.is_file())


--- Hindi ---
【पूरी तरह काल्पनिक उदाहरण】रोगी थघलऩऋणटउईथझ, मेडिकल रिकॉर्ड 811238458904, आधार 301334291830 और आभा 14346862362955। फोन +91 92800 54383, ईमेल regepallavi@example.test। छआगऊछफणआषध ने हल्के बुखार के लिए आराम की सलाह दी।
[
  {
    "label": "PERSON",
    "text": "अनन्या मेहता",
    "start": 31,
    "end": 43,
    "confidence": 1.0,
    "replacement": "थघलऩऋणटउईथझ",
    "sources": [
      "custom:deny"
    ]
  },
  {
    "label": "ID_NUM",
    "text": "HI-DEMO-708",
    "start": 60,
    "end": 71,
    "confidence": 1.0,
    "replacement": "811238458904",
    "sources": [
      "custom:deny"
    ]
  },
  {
    "label": "ID_NUM",
    "text": "2467 7832 5484",
    "start": 78,
    "end": 92,
    "confidence": 0.95,
    "replacement": "301334291830",
    "sources": [
      "safety_sweep"
    ]
  },
  {
    "label": "ID_NUM",
    "text": "91-0000-0000-0000",
    "start": 100,
    "end": 117,
    "confidence": 1.0,
    "replacement": "14346862362955",
    "sources": [
      "custom:de

## 4. Exercise: audit the zero-leak invariant

Build a compact audit that maps each note to any fabricated identifiers still present. Predict the result before running the answer cell: every list should be empty.


In [4]:
# Exercise answer scaffold
leak_audit = {
    "chinese": [
        value
        for value in chinese.SYNTHETIC_IDENTIFIERS
        if value in chinese_result.deidentified_text
    ],
    **{
        note_name: [
            value
            for value in hindi.SYNTHETIC_IDENTIFIERS_BY_NOTE[note_name]
            if value in result.deidentified_text
        ]
        for note_name, result in hindi_results.items()
    },
}
assert all(not leaked for leaked in leak_audit.values())
workspace.cleanup()
leak_audit


{'chinese': [], 'hindi': [], 'hinglish': []}

## Pitfall and live extension

A common mistake is checking only the model's entity count. A detector can return entities while still leaving a known direct identifier in the output, so keep the explicit fail-closed checks.

To run the shipped checkpoints instead of the no-download tutorial loader, install `openmed[hf]` and execute the companion scripts directly from the repository root:

```bash
uv run python examples/deid_chinese_clinical_note.py
uv run python examples/deid_hindi_hinglish_note.py
```

Optional extension: add another clearly fabricated, site-specific identifier to a copied note, add a matching local recognizer term or pattern, and extend the check tuple before processing any real data. Keep jurisdictional identifiers such as Aadhaar and ABHA on their shipped, validated recognizer path.
